In [ ]:
pip install bokeh

In [ ]:
"""
build_recurrence_widget.py
--------------------------
Standalone interactive viewer for one trial's gaze recurrence structure.

LEFT  : z-scored gaze angle vs time (scatter)
RIGHT : recurrence plot (scatter of (i, j) where |z_i - z_j| <= epsilon)

Click any point in EITHER plot:
    -> the time index t_i becomes the "anchor"
    -> on the LEFT, every other time point j with |z_j - z_i| <= eps is highlighted
    -> on the RIGHT, every recurrent point in row i or column i is highlighted
       (these are exactly the same set of partners, just shown in matrix form)

Designed to drop in next to the existing RQA pipeline; uses the same
GAZE_COL, the same z-scoring, and the same epsilon convention.
"""

import os
import numpy as np
import pandas as pd

from bokeh.plotting import figure, output_file, save
from bokeh.layouts import row, column
from bokeh.models import (
    ColumnDataSource, CustomJS, Div, HoverTool, Span,
)
from bokeh.io import show as bokeh_show
from bokeh.resources import INLINE


GAZE_COL = "gaze angle [deg]"


# -------------------------------------------------------------------
# Shared JS for linked highlighting (same logic regardless of which
# plot was clicked; only the way `target_idx` is derived differs).
# -------------------------------------------------------------------
_HIGHLIGHT_FN = r"""
function applyHighlight(target_idx) {
    // ---- LEFT: gaze-z scatter ----
    const z_target = z[target_idx];

    const lc = left.data.color;
    const la = left.data.alpha;
    const ls = left.data.size;
    for (let i = 0; i < lc.length; i++) {
        if (i === target_idx) {
            lc[i] = anchor_color;
            la[i] = 1.0;
            ls[i] = 12;
        } else if (Math.abs(z[i] - z_target) <= eps) {
            lc[i] = hi_color;
            la[i] = 0.95;
            ls[i] = 7;
        } else {
            lc[i] = base_color;
            la[i] = 0.18;
            ls[i] = 4;
        }
    }
    left.change.emit();

    // ---- RIGHT: recurrence scatter ----
    const ri = right.data.i;
    const rj = right.data.j;
    const rc = right.data.color;
    const ra = right.data.alpha;
    const rs = right.data.size;
    for (let k = 0; k < ri.length; k++) {
        if (ri[k] === target_idx || rj[k] === target_idx) {
            rc[k] = hi_color;
            ra[k] = 1.0;
            rs[k] = 5;
        } else {
            rc[k] = base_color;
            ra[k] = 0.10;
            rs[k] = 2;
        }
    }
    right.change.emit();

    // ---- Crosshair spans on the recurrence plot ----
    vline.location = target_idx;
    hline.location = target_idx;
    vline.visible = true;
    hline.visible = true;

    // ---- Anchor line on the time-series plot ----
    tline.location = target_idx;
    tline.visible = true;

    // ---- Update info banner ----
    let n_partners = 0;
    for (let i = 0; i < z.length; i++) {
        if (i !== target_idx && Math.abs(z[i] - z_target) <= eps) n_partners++;
    }
    info.text =
        "<div style='font-family:ui-monospace,Menlo,monospace;font-size:13px;color:#222;'>" +
        "<b>anchor</b> &nbsp; t = " + target_idx +
        " &nbsp;|&nbsp; z = " + z_target.toFixed(3) +
        " &nbsp;|&nbsp; <b>" + n_partners + "</b> recurrent partners (within ε = " +
        eps.toFixed(3) + ")</div>";
}
"""


def build_linked_recurrence_widget(
    csv_path,
    epsilon=0.07,
    output_html="recurrence_widget.html",
    show=False,
):
    """
    Build an interactive linked-brushing visualization for a single trial.

    Parameters
    ----------
    csv_path : str
        Path to a trial CSV that contains GAZE_COL.
    epsilon : float
        Recurrence radius in z-score units (matches the RQA pipeline).
    output_html : str or None
        If given, save the standalone HTML file to this path.
    show : bool
        If True, also open the saved HTML in the default browser.

    Returns
    -------
    bokeh layout object
    """
    # ----------------------------------------------------------------
    # Load + z-score (same recipe used in the RQA pipeline)
    # ----------------------------------------------------------------
    df = pd.read_csv(csv_path)
    if GAZE_COL not in df.columns:
        raise KeyError(f"'{GAZE_COL}' not found in {csv_path}")

    gaze = df[GAZE_COL].to_numpy(dtype=float)
    gaze_clean = gaze[~np.isnan(gaze)]
    if gaze_clean.size < 2:
        raise ValueError(f"Not enough valid data in {csv_path}")

    mu = gaze_clean.mean()
    sd = gaze_clean.std(ddof=1)
    if sd == 0 or np.isnan(sd):
        raise ValueError(f"Zero/NaN std in {csv_path}; cannot z-score.")

    gaze_z = (gaze_clean - mu) / sd
    N = gaze_z.size
    t = np.arange(N)

    # ----------------------------------------------------------------
    # Recurrence matrix (1D phase space, |z_i - z_j| <= eps)
    # Drop self-recurrence so the diagonal doesn't dominate visually.
    # ----------------------------------------------------------------
    dist = np.abs(gaze_z[:, None] - gaze_z[None, :])
    R = dist <= epsilon
    np.fill_diagonal(R, False)
    rec_i, rec_j = np.where(R)

    n_rec = int(R.sum())
    rr = n_rec / max(R.size - N, 1)  # exclude diagonal from total

    if len(rec_i) > 1_500_000:
        print(
            f"[warn] {len(rec_i):,} recurrent points; render may be slow. "
            "Consider downsampling the trial or raising epsilon."
        )

    # ----------------------------------------------------------------
    # Sources
    # ----------------------------------------------------------------
    base_color   = "#2b3a55"   # deep navy
    hi_color     = "#e63946"   # vivid red
    anchor_color = "#000000"

    left_source = ColumnDataSource(dict(
        idx=t.tolist(),
        t=t.tolist(),
        z=gaze_z.tolist(),
        color=[base_color] * N,
        alpha=[0.55] * N,
        size=[5] * N,
    ))

    right_source = ColumnDataSource(dict(
        i=rec_i.tolist(),
        j=rec_j.tolist(),
        color=[base_color] * len(rec_i),
        alpha=[0.35] * len(rec_i),
        size=[2] * len(rec_i),
    ))

    # ----------------------------------------------------------------
    # Figures
    # ----------------------------------------------------------------
    p_left = figure(
        width=620, height=540,
        title=f"z-scored gaze · ε = {epsilon}",
        x_axis_label="time index (sample)",
        y_axis_label="gaze angle (z-score)",
        tools="pan,wheel_zoom,box_zoom,reset,tap,save",
        active_scroll="wheel_zoom",
        output_backend="webgl",
        background_fill_color="#fafafa",
    )
    p_left.scatter(
        x="t", y="z",
        source=left_source,
        color="color", alpha="alpha", size="size",
        line_color=None,
    )
    p_left.line(t.tolist(), gaze_z.tolist(),
                color="#bbbbbb", line_width=0.8, alpha=0.6, level="underlay")

    p_right = figure(
        width=560, height=540,
        title=f"recurrence plot · {n_rec:,} pts · RR ≈ {rr:.3f}",
        x_axis_label="i (sample)",
        y_axis_label="j (sample)",
        tools="pan,wheel_zoom,box_zoom,reset,tap,save",
        active_scroll="wheel_zoom",
        match_aspect=True,
        output_backend="webgl",
        background_fill_color="#fafafa",
    )
    p_right.scatter(
        x="i", y="j",
        source=right_source,
        color="color", alpha="alpha", size="size",
        line_color=None,
    )

    # Crosshair spans (hidden until first click)
    vline = Span(location=0, dimension="height", line_color="#e63946",
                 line_dash="dashed", line_width=1.2, visible=False)
    hline = Span(location=0, dimension="width",  line_color="#e63946",
                 line_dash="dashed", line_width=1.2, visible=False)
    p_right.add_layout(vline)
    p_right.add_layout(hline)

    tline = Span(location=0, dimension="height", line_color="#000000",
                 line_dash="dotted", line_width=1.2, visible=False)
    p_left.add_layout(tline)

    # ----------------------------------------------------------------
    # Hover tooltips
    # ----------------------------------------------------------------
    p_left.add_tools(HoverTool(
        tooltips=[("t", "@t"), ("z", "@z{0.000}")],
        mode="mouse",
    ))
    p_right.add_tools(HoverTool(
        tooltips=[("i", "@i"), ("j", "@j")],
        mode="mouse",
    ))

    # ----------------------------------------------------------------
    # Info banner
    # ----------------------------------------------------------------
    info = Div(text=(
        "<div style='font-family:ui-monospace,Menlo,monospace;"
        "font-size:13px;color:#666;'>"
        "click any point to anchor &mdash; partners highlight in both panels"
        "</div>"
    ), width=1180)

    # ----------------------------------------------------------------
    # Linked-brushing callbacks
    # ----------------------------------------------------------------
    js_args = dict(
        left=left_source,
        right=right_source,
        z=gaze_z.tolist(),
        eps=float(epsilon),
        base_color=base_color,
        hi_color=hi_color,
        anchor_color=anchor_color,
        vline=vline, hline=hline, tline=tline,
        info=info,
    )

    left_cb = CustomJS(args=js_args, code=_HIGHLIGHT_FN + r"""
        const sel = left.selected.indices;
        if (sel.length === 0) return;
        const target_idx = left.data.idx[sel[0]];
        applyHighlight(target_idx);
    """)
    right_cb = CustomJS(args=js_args, code=_HIGHLIGHT_FN + r"""
        const sel = right.selected.indices;
        if (sel.length === 0) return;
        // anchor on the row-index 'i' of the clicked recurrence cell
        const target_idx = right.data.i[sel[0]];
        applyHighlight(target_idx);
    """)
    left_source.selected.js_on_change("indices", left_cb)
    right_source.selected.js_on_change("indices", right_cb)

    # ----------------------------------------------------------------
    # Header
    # ----------------------------------------------------------------
    header = Div(text=f"""
        <div style='font-family:ui-sans-serif,system-ui,sans-serif;'>
            <h2 style='margin:0 0 4px 0;font-weight:600;letter-spacing:-0.01em;'>
                Linked recurrence viewer
            </h2>
            <div style='color:#555;font-size:13px;'>
                <code>{os.path.basename(csv_path)}</code>
                &nbsp;·&nbsp; N = {N:,}
                &nbsp;·&nbsp; ε = {epsilon}
                &nbsp;·&nbsp; recurrent points = {n_rec:,}
            </div>
        </div>
    """, width=1180)

    layout = column(header, info, row(p_left, p_right))

    if output_html:
        output_file(output_html, title="Linked Recurrence Viewer", mode="inline")
        save(layout, resources=INLINE)
    if show:
        bokeh_show(layout)
    return layout


# -------------------------------------------------------------------
# CLI entry point — interactive file picker
# -------------------------------------------------------------------
def _pick_file_via_dialog(base_dir):
    """Open a native OS file-picker dialog (tkinter).

    Returns
    -------
    str | None | False
        - str : absolute path the user chose
        - None: the user cancelled the dialog
        - False: tkinter / display unavailable -> caller should fall back
    """
    try:
        import tkinter as tk
        from tkinter import filedialog
    except ImportError:
        return False

    try:
        root = tk.Tk()
    except Exception:
        # e.g. no $DISPLAY on a headless Linux box
        return False

    try:
        root.withdraw()                         # hide the empty parent window
        root.attributes("-topmost", True)       # bring the dialog to the front
        path = filedialog.askopenfilename(
            title="Select a trial CSV",
            initialdir=os.path.abspath(base_dir),
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")],
        )
    finally:
        root.destroy()

    return path if path else None  # askopenfilename returns "" on cancel


def _prompt_for_file(csvs, base_dir):
    """Terminal fallback: numbered list + substring search.

    Used only when a GUI file dialog is unavailable.
    """
    print(f"\nCSV files in '{base_dir}':\n")
    width = len(str(len(csvs)))
    for i, name in enumerate(csvs, start=1):
        print(f"  [{i:>{width}}] {name}")

    while True:
        try:
            choice = input(
                f"\nSelect a file (1-{len(csvs)}), or type a substring "
                "(blank/q to quit): "
            ).strip()
        except (EOFError, KeyboardInterrupt):
            print("\nAborted.")
            return None

        if choice == "" or choice.lower() in {"q", "quit", "exit"}:
            print("Aborted.")
            return None

        if choice.isdigit():
            n = int(choice)
            if 1 <= n <= len(csvs):
                return os.path.join(base_dir, csvs[n - 1])
            print(f"  -> out of range; pick a number between 1 and {len(csvs)}")
            continue

        matches = [c for c in csvs if choice.lower() in c.lower()]
        if len(matches) == 1:
            return os.path.join(base_dir, matches[0])
        if len(matches) == 0:
            print("  -> no filename contains that text; try again")
        else:
            print(f"  -> '{choice}' is ambiguous ({len(matches)} matches):")
            for m in matches[:10]:
                print(f"       {m}")
            if len(matches) > 10:
                print(f"       ... and {len(matches) - 10} more")


if __name__ == "__main__":
    BASE_DIR = "walk_segmented_csvs"
    EPSILON = 0.07
    OUT = "recurrence_widget.html"

    if not os.path.isdir(BASE_DIR):
        raise FileNotFoundError(
            f"Directory '{BASE_DIR}' not found. "
            "Run this from the same folder as your RQA pipeline."
        )

    csvs = sorted(f for f in os.listdir(BASE_DIR) if f.lower().endswith(".csv"))
    if not csvs:
        raise RuntimeError(f"No CSVs in {BASE_DIR}")

    # Try the native file dialog first; fall back to the terminal picker.
    chosen = _pick_file_via_dialog(BASE_DIR)
    if chosen is False:
        print("(GUI file dialog unavailable; using terminal picker)")
        chosen = _prompt_for_file(csvs, BASE_DIR)

    if chosen is None:
        print("No file selected.")
        raise SystemExit(0)

    build_linked_recurrence_widget(
        chosen,
        epsilon=EPSILON,
        output_html=OUT,
        show=False,
    )
    print(f"\nWrote {OUT} for {os.path.basename(chosen)}")


Wrote recurrence_widget.html for Subject-02_Hill-Condition_Trial-11_Walk-Dir-Up.csv


: 